# L05 · 收益率 + 涨停识别

**预计时长**：60 min | **难度**：⭐⭐⭐ | **前置**：L04

## 本节目标
1. 彻底搞清 `shift / diff / pct_change` 三者关系
2. 日收益率、累计收益率、年化收益率（252 交易日/年）
3. 代码识别涨停日（含一字涨停）
4. 复利 vs 算术累加的区别

## 第 1 段：金融概念

### 三种"收益率"
| 类型 | 公式 | 用途 |
|------|------|------|
| 单日收益率 | (close - prev_close) / prev_close | 日度分析 |
| 累计收益率 | (final_close - initial_close) / initial_close | 简单区间统计 |
| 复利累计 | ∏(1 + r_t) - 1 | **正确**做法 |
| 年化收益率 | (1 + 累计) ** (252/N) - 1 | 跨期比较 |

### 复利 vs 算术累加（新手陷阱）
- ❌ 累计 = sum(日收益率) — 错的！会低估
- ✅ 累计 = prod(1 + 日收益率) - 1 — 正确

### 涨停识别（L01 深化版）
- **普通涨停**：chg_pct >= 9.9%（主板）
- **一字涨停**：open == high == low == close 且 chg_pct >= 9.9%（最强势）
- **秒板**：开盘即涨停，但日内可能有波动（数据上看就是 open == close）

一字涨停是**最强信号**，意味着开盘瞬间就封板，没人能买到（除非早上挂单）。

In [ ]:
import sys
from pathlib import Path

# 自动定位 phase1_foundation 目录 + project root（兼容两种 jupyter 启动位置）
_cwd = Path.cwd()
_p1 = _cwd if (_cwd / '_data.py').exists() else (_cwd / 'learning' / 'phase1_foundation')
sys.path.insert(0, str(_p1))
_proj = _p1.parent.parent if _p1.name == 'phase1_foundation' else _p1
if (_proj / 'qtrader' / '__init__.py').exists():
    sys.path.insert(0, str(_proj))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import _style
from _data import get_stock_data

In [ ]:
byd = get_stock_data('002594')
byd = byd.set_index('date')
byd.head(3)

## 第 2 段：`shift / diff / pct_change` 三剑客

In [ ]:
# shift: 整列平移
byd['prev_close'] = byd['close'].shift(1)        # 前一日
byd['next_close'] = byd['close'].shift(-1)       # 后一日（小心未来函数！）

# diff: 一阶差分（当前 - 前一）
byd['abs_change'] = byd['close'].diff(1)         # close - prev_close

# pct_change: 百分比变化
byd['ret'] = byd['close'].pct_change(1)          # = (close - prev_close) / prev_close

byd[['close', 'prev_close', 'abs_change', 'ret']].head()

In [ ]:
# 三者关系：
# diff(1)   = close - shift(1)
# pct_change(1) = diff(1) / shift(1)
# pct_change 是最常用的，因为收益率本身是无量纲的（百分比）

## 第 3 段：累计收益率 `cumprod`

In [ ]:
# 复利累计收益 = ∏(1 + r_t) - 1
byd['cum_ret'] = (1 + byd['ret']).cumprod() - 1

# 对比算术累加（错的！）
byd['cum_ret_wrong'] = byd['ret'].cumsum()

fig, ax = plt.subplots(figsize=(12, 5))
byd['cum_ret'].plot(ax=ax, label='复利（正确）', linewidth=2)
byd['cum_ret_wrong'].plot(ax=ax, label='算术累加（错）', linestyle='--')
ax.axhline(0, color='black', alpha=0.3)
ax.legend()
ax.set_title('比亚迪 累计收益率：复利 vs 算术累加')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x*100:.0f}%'))
plt.show()

## 第 4 段：年化收益率（252 交易日/年）

In [ ]:
def annualized_return(df: pd.DataFrame, col: str = 'close') -> float:
    """计算年化收益率。

    公式：(final/initial) ** (252/N) - 1
    """
    n_days = len(df)
    initial = df[col].iloc[0]
    final = df[col].iloc[-1]
    return (final / initial) ** (252 / n_days) - 1

for code, name in [("002594", "比亚迪"), ("002602", "世纪华通"), ("002624", "完美世界")]:
    df = get_stock_data(code).set_index('date')
    ann = annualized_return(df)
    cum = (df['close'].iloc[-1] / df['close'].iloc[0]) - 1
    print(f"{name:<8} {code}  累计 {cum*100:>7.2f}%  年化 {ann*100:>7.2f}%")

## 第 5 段：代码识别涨停日（含一字板）

In [ ]:
def find_limit_ups(df: pd.DataFrame, threshold: float = 0.099) -> pd.DataFrame:
    """识别涨停日 + 一字板。

    ⚠️ 严格涨停识别应用「不复权」数据（raw）。本函数用 qfq 数据，
    除权日可能轻微误判（qfq 会把除权日"缺口"补回去，导致 chg_pct
    虚高）。教学场景可接受，实战回测请传 adjust="" 的数据。

    Returns:
        DataFrame index=date，列 [close, prev_close, chg_pct, is_yizi]
        is_yizi=True 表示一字涨停（开盘=最高=最低=收盘）
    """
    d = df.copy()
    d['prev_close'] = d['close'].shift(1)
    d['chg_pct'] = d['close'] / d['prev_close'] - 1
    limit_up = d[d['chg_pct'] >= threshold].copy()
    # 一字板判定：4 价几乎相等
    eps = 0.001  # 千分之一容差
    limit_up['is_yizi'] = (
        (abs(limit_up['open'] - limit_up['high']) < eps * limit_up['close']) &
        (abs(limit_up['high'] - limit_up['low']) < eps * limit_up['close']) &
        (abs(limit_up['low'] - limit_up['close']) < eps * limit_up['close'])
    )
    return limit_up

byd_lu = find_limit_ups(get_stock_data('002594').set_index('date'))
print(f"比亚迪涨停 {len(byd_lu)} 次，其中一字板 {byd_lu['is_yizi'].sum()} 次")
byd_lu[byd_lu['is_yizi']].head()

## 第 6 段：随堂小练

### 小练：三股累计收益率排名

In [ ]:
# TODO: 你的代码（约 6 行）
# 对 002594 / 002602 / 002624 三只股票
# 用 (1+ret).cumprod()-1 算 2024 全年累计收益率
# 按收益从高到低排名打印

## 第 7 段：小测（5 题）

在 notebook 里直接答，答完告诉我答案：
1. `shift(1)` 和 `shift(-1)` 哪个有"未来函数"风险？
2. `pct_change()` 和 `diff() / shift(1)` 哪个返回值无量纲？
3. 主板涨停阈值是 9.9% 还是 10%？为什么不是整数？
4. 一字涨停的判定条件用 `open == high == low == close` 对吗？为什么实际代码要加 `eps`？
5. 252 这个数字怎么来的？

答完我给反馈。

## 第 8 段：课后练习 + 下节预告

### 📝 `exercises/ex05.py`
1. 写 `find_one_word_limit_ups(df, threshold=0.099, eps=0.001)` 返回一字涨停日列表
2. 写 `annualized_return_with_costs(df, cost_bps=10)`：扣除每笔交易成本（L06 讲）后的年化
3. 对三股计算 2022-2024 每年"涨停日数 + 年化收益率"，做一张 3×3 的透视表

### 🔮 下节 L06：统计基础 + 交易成本
量化必备的"风险度量"：均值、方差、标准差、相关系数。同时讲清交易成本如何拖垮策略。

## 第 9 段：Jupyter tip 🔧
- `Series.clip(lower=-0.11, upper=0.11)`：把涨跌幅限制在 ±11%（防涨停日异常）
- `pd.Timedelta(days=1)`：时间差，比手写 `86400` 秒清晰
- `df.rolling(20).apply(my_func)`：自定义滚动函数（但慢，能用向量化就别用 apply）